# BGK FPC PLN — MVP dashboard

Pierwszy MVP. Trzy sekcje:

1. **Outstanding timeline** — cumulative FPC PLN outstanding (mld PLN) od 2020-06 → dzisiaj, z `v_bgk_outstanding_timeline`.
2. **Spread vs POLGB** — per-auction spread BGK-vs-POLGB curve w bp, z `v_bgk_auction_spread` (Milestone B).
3. **Recent auctions** — ostatnie 20 aukcji z B/C, sold, yield, spread.

Scope: tylko FPC PLN. Źródło danych: Supabase project shared z CETO_DOWNLOADER.

**Setup:** `pip install -r requirements-notebook.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org`, ustaw `SUPABASE_URL` i `SUPABASE_SERVICE_ROLE_KEY` w `.env` lub w shell przed `jupyter lab`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import requests
from IPython.display import display

# Load .env if present; else rely on shell-set env vars.
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

pio.renderers.default = 'notebook_connected'

SUPABASE_URL = os.environ['SUPABASE_URL'].rstrip('/')
# Normalise: tolerate operator pasting the PostgREST endpoint by mistake.
for suffix in ('/rest/v1', '/rest'):
    if SUPABASE_URL.endswith(suffix):
        SUPABASE_URL = SUPABASE_URL[: -len(suffix)]
        break
SUPABASE_KEY = os.environ['SUPABASE_SERVICE_ROLE_KEY']

HEADERS = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type': 'application/json',
}
PAGE_SIZE = 1000


def fetch_view(name: str, query: str = '?select=*') -> pd.DataFrame:
    """GET a PostgREST view/table, paginating until exhausted."""
    rows: list = []
    for page in range(200):
        offset = page * PAGE_SIZE
        h = {**HEADERS, 'Range-Unit': 'items',
             'Range': f'{offset}-{offset + PAGE_SIZE - 1}'}
        r = requests.get(f'{SUPABASE_URL}/rest/v1/{name}{query}',
                         headers=h, timeout=60)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            break
    return pd.DataFrame(rows)


print(f'Connected to {SUPABASE_URL}')

## 1. Outstanding FPC PLN — running total over time

Step chart: każda emisja podbija outstanding, każdy wykup go obniża. Z `v_bgk_outstanding_timeline` przefiltrowane do `program='FPC' AND currency='PLN'`.

In [ ]:
df_timeline = fetch_view(
    'v_bgk_outstanding_timeline',
    '?program=eq.FPC&currency=eq.PLN&select=*&order=event_date.asc',
)
df_timeline['event_date'] = pd.to_datetime(df_timeline['event_date'])
# Stored in PLN; display in mld.
df_timeline['outstanding_bn'] = df_timeline['running_outstanding'].astype(float) / 1e9

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_timeline['event_date'],
    y=df_timeline['outstanding_bn'],
    mode='lines',
    line_shape='hv',  # step chart - issuance steps up, redemption steps down
    name='FPC PLN outstanding',
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.18)',
    line=dict(color='rgb(31, 119, 180)', width=2),
    hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} mld PLN<extra></extra>',
))
fig.update_layout(
    title='BGK FPC PLN outstanding (mld PLN)',
    xaxis_title=None,
    yaxis_title='Outstanding (mld PLN)',
    hovermode='x unified',
    height=420,
    template='plotly_white',
)
fig.show()

latest = df_timeline.iloc[-1]
print(f"Latest change: {latest['event_date'].date()} → "
      f"{latest['outstanding_bn']:.1f} mld PLN "
      f"({len(df_timeline)} change events since {df_timeline['event_date'].iloc[0].date()})")

## 2. Spread vs POLGB curve — per series over time

Per-auction spread w bp pomiędzy BGK FPC yield (z PDF-ów) a interpolowaną POLGB krzywą na tym samym tenorze. Pozytywny spread = BGK płaci premium nad SP.

Floatery (FPC0332) wykluczone — ich spread liczony w DM-space jakościowo inny, na osobnym wykresie później.
FPC1140 (~14Y) wykluczony — dłuższy niż najdłuższy POLGB w krzywej (`spread_bp IS NULL` per Milestone B decision).

In [ ]:
# url-encoded 'stałe' = sta%C5%82e (UTF-8 ł = 0xC5 0x82). PostgREST query param needs encoding.
df_spread = fetch_view(
    'v_bgk_auction_spread',
    '?bgk_coupon_kind=eq.sta%C5%82e&spread_bp=not.is.null'
    '&series=like.FPC*&select=*&order=auction_date.asc',
)
df_spread['auction_date'] = pd.to_datetime(df_spread['auction_date'])
df_spread['spread_bp'] = df_spread['spread_bp'].astype(float)
df_spread['tenor_years'] = df_spread['tenor_years'].astype(float)

fig = go.Figure()
for series in sorted(df_spread['series'].unique()):
    sub = df_spread[df_spread['series'] == series]
    fig.add_trace(go.Scatter(
        x=sub['auction_date'],
        y=sub['spread_bp'],
        mode='lines+markers',
        name=series,
        customdata=sub['tenor_years'],
        hovertemplate=(f'{series}<br>'
                       '%{x|%Y-%m-%d}<br>'
                       'spread: %{y:.1f} bp<br>'
                       'tenor: %{customdata:.1f} y<extra></extra>'),
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    title='BGK FPC PLN spread vs POLGB curve (bp)',
    xaxis_title=None,
    yaxis_title='Spread (bp)',
    hovermode='x unified',
    height=460,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

summary = (df_spread.groupby('series')['spread_bp']
           .agg(['count', 'mean', 'min', 'max'])
           .rename(columns={'count': 'n_auctions', 'mean': 'avg_bp',
                            'min': 'min_bp', 'max': 'max_bp'}))
summary[['avg_bp', 'min_bp', 'max_bp']] = summary[['avg_bp', 'min_bp', 'max_bp']].round(1)
print('\nSpread summary per series:')
display(summary)

## 3. Recent auctions — last 20

Per-auction snapshot z `v_bgk_auction_metrics` JOIN `v_bgk_auction_spread`. Kolumny:
- `demand_mln` / `sold_mln` — popyt vs sprzedaż (mln PLN)
- `b/c` — bid-to-cover (>2 = mocny popyt)
- `yield_%` — yield z aukcji (NULL dla floaterów)
- `addl_sale_mln` — sprzedaż dodatkowa (top-up po main)
- `spread_bp` — vs POLGB curve (NULL gdy tenor poza zakresem krzywej)

In [ ]:
df_recent_m = fetch_view(
    'v_bgk_auction_metrics',
    '?series=like.FPC*&select=auction_date,series,isin,'
    'demand_total_mln,sold_total_mln,bid_to_cover,yield_pct,'
    'additional_sale_mln,outstanding_after_mln&order=auction_date.desc,series.asc',
)
df_recent_s = fetch_view(
    'v_bgk_auction_spread',
    '?select=auction_date,series,spread_bp,tenor_years',
)
df = df_recent_m.merge(df_recent_s, on=['auction_date', 'series'], how='left')
df = df.head(20)

for c in ['demand_total_mln', 'sold_total_mln', 'bid_to_cover', 'yield_pct',
          'additional_sale_mln', 'outstanding_after_mln',
          'spread_bp', 'tenor_years']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df_show = df.rename(columns={
    'auction_date': 'date',
    'demand_total_mln': 'demand_mln',
    'sold_total_mln': 'sold_mln',
    'bid_to_cover': 'b/c',
    'yield_pct': 'yield_%',
    'additional_sale_mln': 'addl_sale_mln',
    'outstanding_after_mln': 'outstanding_mln',
    'tenor_years': 'tenor_y',
})

display(df_show.style.format({
    'demand_mln':      '{:,.0f}',
    'sold_mln':        '{:,.0f}',
    'b/c':             '{:.2f}',
    'yield_%':         '{:.3f}',
    'addl_sale_mln':   '{:,.0f}',
    'outstanding_mln': '{:,.0f}',
    'spread_bp':       '{:.1f}',
    'tenor_y':         '{:.1f}',
}, na_rep='—').hide(axis='index'))